## Random Forest Feature Engineering

This notebook:
1. Loads flight and weather raw data from `1_downloaddata/raw`
2. Creates a `target` label (`Cancelled`, `Delayed`, `On time`)
3. Removes leakage columns
4. Builds hourly weather features
5. Merges weather to flights by origin/date/departure hour
6. Saves a cleaned modeling dataset

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import holidays

RAW_DIR = Path("1_downloaddata/raw")
CLEAN_FILE = Path("2_intermediate/cleaned_flights_weather.parquet")

def read_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported file type: {path}")

In [2]:
from pathlib import Path

# Resolve raw data path robustly from current working directory
HERE = Path.cwd()

candidate_dirs = [

    HERE.parent.parent / "1_download_data" / "raw",
    HERE.parent.parent / "1_donload_data" / "raw",
]

RAW_DIR = next((p for p in candidate_dirs if p.exists()), None)

if RAW_DIR is None:
    raise FileNotFoundError(
        f"Could not find raw data folder from cwd={HERE}. "
        "Checked: 1_downloaddata/raw, 1_download_data/raw, 1_donload_data/raw "
        "in cwd, parent, and grandparent."
    )

print("Using RAW_DIR:", RAW_DIR)

Using RAW_DIR: /home/arv020/DSC288R_DataScienceCapstone/1_download_data/raw


### Locate flight and weather files

We scan the raw folder for parquet/csv files and identify the weather file by name.

In [4]:
raw_files = sorted(list(RAW_DIR.glob("*.parquet")) + list(RAW_DIR.glob("*.csv")))
if len(raw_files) < 2:
    raise FileNotFoundError(f"Expected at least 2 files in {RAW_DIR}, found {len(raw_files)}")

weather_candidates = [f for f in raw_files if "weather" in f.name.lower()]
if not weather_candidates:
    raise FileNotFoundError("Could not identify weather file (filename should include 'weather').")

WEATHER_FILE = weather_candidates[0]
RAW_FILE = next(f for f in raw_files if f != WEATHER_FILE)

df = read_table(RAW_FILE)
print("Flight shape:", df.shape)


Flight shape: (29193782, 62)


### Create label and remove leakage features

- `target = Cancelled` if cancelled flight  
- `target = Delayed` if departure delay >= 15 min  
- Otherwise `On time`

Then we remove columns that would leak post-departure outcomes into modeling.

In [5]:
df["FlightDate"] = pd.to_datetime(df["FlightDate"], errors="coerce")
df["CRSDepTime"] = pd.to_numeric(df["CRSDepTime"], errors="coerce").fillna(0).astype(int)

conditions = [
    df["Cancelled"] == True,
    df["DepDelayMinutes"] >= 15
]
choices = ["Cancelled", "Delayed"]
df["target"] = np.select(conditions, choices, default="On time")

#dropping data leakage columns
df = df.drop(
    columns=[
        "DepTime","DepDelay","DepDelayMinutes","DepDel15","DepartureDelayGroups",
        "ArrTime","ArrDelay","ArrDelayMinutes","ArrDel15","ArrivalDelayGroups",
        "Cancelled","Diverted","DivAirportLandings","TaxiOut","TaxiIn","WheelsOff",
        "WheelsOn","ActualElapsedTime","AirTime", "CRSArrTime", "CRSElapsedTime", "year"
    ],
    errors="ignore"
)

#Dropping duplicate columns
df = df.drop(columns = ["Flight_Number_Marketing_Airline", "OriginAirportSeqID", "DestAirportSeqID", 
                        "IATA_Code_Operating_Airline", "Operating_Airline", "IATA_Code_Marketing_Airline",
                        "OriginStateName", "DestStateName"
                       ])


if "Year" in df.columns:
    df = df[df["Year"] != 2020]

print("Flights after engineering:", df.shape)

Flights after engineering: (24171385, 33)


In [6]:
# holidays
us_holidays = holidays.US()
print("Holidays feature started")

# Is Holiday (0/1)
df["is_holiday"] = df["FlightDate"].dt.date.apply(
    lambda x: 1 if x in us_holidays else 0)

Holidays feature started


In [7]:
df["date"] = df["FlightDate"].dt.date
df['dep_hour'] = df['CRSDepTime'] // 100


In [8]:
df["is_morning_peak"] = df["dep_hour"].between(6, 9).astype(int)

# Evening peak: 17–23
df["is_evening_peak"] = df["dep_hour"].between(17, 23).astype(int)

In [9]:
df.head()

,FlightDate,Airline,Origin,Dest,CRSDepTime,Distance,Year,Quarter,Month,DayofMonth,...,DestWac,DepTimeBlk,ArrTimeBlk,DistanceGroup,target,is_holiday,date,dep_hour,is_morning_peak,is_evening_peak
0,2018-01-23,Endeavor Air Inc.,ABY,ATL,1202,145.0,2018,1,1,23,...,34,1200-1259,1300-1359,1,On time,0,2018-01-23,12,0,0
1,2018-01-24,Endeavor Air Inc.,ABY,ATL,1202,145.0,2018,1,1,24,...,34,1200-1259,1300-1359,1,On time,0,2018-01-24,12,0,0
2,2018-01-25,Endeavor Air Inc.,ABY,ATL,1202,145.0,2018,1,1,25,...,34,1200-1259,1300-1359,1,On time,0,2018-01-25,12,0,0
3,2018-01-26,Endeavor Air Inc.,ABY,ATL,1202,145.0,2018,1,1,26,...,34,1200-1259,1300-1359,1,On time,0,2018-01-26,12,0,0
4,2018-01-27,Endeavor Air Inc.,ABY,ATL,1400,145.0,2018,1,1,27,...,34,1400-1459,1500-1559,1,On time,0,2018-01-27,14,0,0


In [10]:
CLEAN_FILE = Path("../../1_download_data/model_ready/RF_V1.parquet")
# Save
CLEAN_FILE.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(CLEAN_FILE, index=False)
print(f"Saved cleaned dataset to {CLEAN_FILE}")

Saved cleaned dataset to ../../1_download_data/model_ready/RF_V1.parquet


In [11]:
df.shape

(24171385, 38)